[![pythonista.io](img/pythonista.png)](https://www.pythonista.io)

# *SBOM*, *provenance* y *attestations*.

Este bloque parte de la *SBOM* de *runners* y la extiende hacia la trazabilidad del artefacto producido por nuestro propio *pipeline*.

## Objetivos.

- Entender para qué sirve una *SBOM*.
- Relacionar *build* reproducible con *provenance*.
- Preparar políticas de verificación antes del despliegue.

## Conceptos y siglas clave.



- **SBOM** (*Software Bill of Materials*): inventario estructurado de componentes (paquetes, versiones, dependencias y metadatos) que participaron en un artefacto.

- **Provenance**: evidencia del origen y proceso de construcción de un artefacto (quién, cómo, cuándo y con qué insumos se construyó).

- **Attestation**: documento firmado que declara una propiedad verificable del artefacto (por ejemplo, su SBOM o el resultado de una política).

- **Digest**: identificador inmutable de una imagen (por ejemplo, `sha256:...`), usado para referenciar exactamente el mismo contenido.

- **OIDC** (*OpenID Connect*): mecanismo de identidad federada usado para autenticación sin llaves estáticas en workflows.

- **Keyless signing**: firma criptográfica sin almacenar llaves privadas en el repositorio; se apoya en identidad federada y certificados efímeros.

- **Rekor**: registro de transparencia de *Sigstore* donde se almacena evidencia verificable de firmas y attestations.

- **Syft**: herramienta para generar SBOM de archivos, imágenes y artefactos.

- **Cosign**: herramienta para firmar y verificar imágenes y attestations en registros de contenedores.



Idea práctica: *SBOM* describe "qué contiene" el artefacto, *provenance* explica "cómo se construyó" y *attestation* permite "probarlo criptográficamente".


## Por qué importa la trazabilidad.

Una *SBOM* describe qué componentes participaron en la construcción de un artefacto. *GitHub* publica *SBOM* de sus *runners*; en este curso se da el siguiente paso: necesitamos saber qué dependencias, *base image*, herramientas y procesos intervinieron en la imagen o paquete que después será desplegado.

Sin trazabilidad, una imagen puede existir; con trazabilidad, una imagen puede justificarse.

## *Attestations*: *SBOM* firmada y verificable.

Una *attestation* combina dos conceptos: el contenido de la *SBOM* y una firma criptográfica que acredita quién la generó y cuándo. El resultado es un documento que no solo describe los componentes del artefacto, sino que además puede ser verificado de forma independiente.

*Cosign* puede adjuntar una *attestation* directamente al registro de imágenes, asociada al *digest* de la imagen:

```bash
# adjuntar SBOM como attestation a la imagen
cosign attest --yes \
  --predicate sbom.spdx.json \
  --type spdxjson \
  ghcr.io/mi-org/mi-app@sha256:abc123...

# verificar que la attestation existe y es valida
cosign verify-attestation \
  --type spdxjson \
  --certificate-identity-regexp "https://github.com/mi-org/.*" \
  --certificate-oidc-issuer "https://token.actions.githubusercontent.com" \
  ghcr.io/mi-org/mi-app@sha256:abc123...
```

La verificación confirma tres cosas: que la *attestation* fue generada por un *workflow* de *GitHub Actions*, que el certificado está dentro del período de validez registrado en *Rekor*, y que el contenido no fue modificado después de la firma.

Esto permite implementar políticas de despliegue que rechacen imágenes sin *attestation* verificable, cerrando el ciclo entre construcción y operación.

## *Pipeline* completo: *SBOM*, firma y *attestation*.

El flujo que une *Syft*, *Cosign* y *GitHub Actions* en un solo *job*:

```yaml
jobs:
  sbom-sign:
    runs-on: ubuntu-latest
    permissions:
      contents: read
      packages: write
      id-token: write       # requerido para firma *keyless* con *OIDC*

    steps:
      - uses: actions/checkout@v4

      - name: Instalar Syft
        uses: anchore/sbom-action/download-syft@v0.15.0

      - name: Instalar Cosign
        uses: sigstore/cosign-installer@v3.5.0

      - name: Construir imagen
        run: docker build -t ghcr.io/${{ github.repository }}:${{ github.sha }} .

      - name: Push imagen
        run: docker push ghcr.io/${{ github.repository }}:${{ github.sha }}

      - name: Generar SBOM de la imagen
        run: |
          syft ghcr.io/${{ github.repository }}:${{ github.sha }} \
            -o spdx-json=sbom.spdx.json

      - name: Firmar imagen (keyless)
        run: |
          cosign sign --yes \
            ghcr.io/${{ github.repository }}:${{ github.sha }}

      - name: Adjuntar SBOM como attestation
        run: |
          cosign attest --yes \
            --predicate sbom.spdx.json \
            --type spdxjson \
            ghcr.io/${{ github.repository }}:${{ github.sha }}

      - name: Subir SBOM como artefacto del workflow
        uses: actions/upload-artifact@v4
        with:
          name: sbom
          path: sbom.spdx.json
```

El permiso `id-token: write` es el que habilita a *Cosign* para obtener un token *OIDC* de *GitHub* y firmar sin llaves estáticas. Sin ese permiso, la firma *keyless* falla.

In [ ]:
sbom = ['python==3.11', 'flask==3.0.0', 'gunicorn==22.0.0']

for dependencia in sbom:
    print('componente:', dependencia)

## Cierre de control de cadena de suministro.



Checklist mínimo para declarar el pipeline endurecido:



- `id-token: write` en jobs que firman o autentican a nube con OIDC.

- imagen referenciada por `digest` inmutable para firma y verificación.

- SBOM generada y almacenada como artefacto verificable.

- firma keyless con *Cosign* y attestation asociada al `digest`.

- verificación de firma y attestation antes de promover despliegue.

- actualización continua de acciones y dependencias con *Dependabot*.

- protección de workflows y entornos (`test`/`prod`) con aprobaciones.



Si alguno de estos puntos falla, la política de entrega debe bloquear promoción.


## ¿Qué es OWASP?



**OWASP** significa *Open Worldwide Application Security Project*.



Es una comunidad global, abierta y sin fines de lucro que publica guías prácticas para mejorar la seguridad del software.



En este curso, OWASP se usa como marco de referencia para:



- priorizar riesgos de cadena de suministro en CI/CD;

- justificar controles de hardening en *GitHub Actions*;

- alinear políticas de despliegue con prácticas reconocidas por la industria.


## Referencia OWASP para *GitHub Actions*.



Para reforzar políticas de hardening en este curso, toma como base estas referencias:



- **OWASP CI/CD Security Cheat Sheet**: guía de controles para pipelines (aplica directamente a *GitHub Actions*).

  https://cheatsheetseries.owasp.org/cheatsheets/CI_CD_Security_Cheat_Sheet.html

- **OWASP Top 10 CI/CD Security Risks**: marco de riesgos prioritarios en cadena de entrega.

  https://owasp.org/www-project-top-10-ci-cd-security-risks/



Nota de precisión: OWASP no publica una cheat sheet exclusiva de *GitHub Actions*; se usa la guía de CI/CD y se adapta al contexto de *Actions*.



Mapeo práctico al repositorio:



- riesgos de *poisoned pipeline execution* -> pinning de acciones, revisión de workflows y entornos protegidos;

- riesgos de *insufficient flow control* -> separación `plan`/`apply` y aprobaciones en `prod`;

- riesgos de *improper artifact integrity validation* -> firma con *Cosign*, *SBOM* y *attestations* verificables.


<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>